In [1]:
import torch
import torch.nn.functional as F

from future_idx_selector import FutureIDXSelector, FutureIdxSelectorModelLoader, RandomModel
device = 'cuda:0'


In [2]:
class RandomModel:
    def __call__(self, x, *args, **kwargs):
        x = x[:,:,-1]
        return torch.rand(x.shape, device=x.device).unsqueeze(-1)
    # end
# end

In [ ]:
loader_model = FutureIdxSelectorModelLoader(device)
model = loader_model.load('mlp_3.pt')
model = model.to(torch.bfloat16)
# model = RandomModel()

In [4]:
selector = FutureIDXSelector(model)

In [5]:
# attn = torch.rand(24, dtype=torch.float64, device=device).reshape(1,24)
attn = torch.rand(24, dtype=torch.bfloat16, device=device).reshape(1,24) - 0.5

# print(attn)

with torch.no_grad():
    result = selector.select_future_by_attn(attn)
    print(result)
# end

tensor([[11, 22,  7,  9, 17]], device='cuda:0')


In [4]:
def select_future_by_3(model, met, h):  # (1, Q, 3)
    attn = met[:, :, -1]    # (1, Q)
    index_avail = (attn >0).nonzero(as_tuple=True)[1].reshape(attn.shape[0], -1)    # (1, Q)
    index_avail_3 = index_avail.view(1,-1,1).expand(1,-1,3)
    met_avail = torch.gather(met, 1, index_avail_3)
    scores = model(met_avail).squeeze(-1)
    
    idx = scores.argsort(dim=-1)[:, :h]
    return torch.gather(index_avail, 1, idx)
# end

In [5]:
met = torch.rand(24, dtype=torch.bfloat16, device=device).reshape(1,-1,3) - 0.5

In [19]:
select_future_by_3(model, met, 5)

tensor([[4, 7, 6, 3, 1]], device='cuda:0')